# Week 4: locked baseline, failure analysis, Experiment 1, validated predict.py
Runtime → Change runtime type → **GPU**. Run top to bottom. Everything runs through `reproduce.sh`, the same single command a teammate would use.

Plan: **Part A** (sections 1–4) runs the baseline, analysis, demo and tests. Then you pre-register Experiment 1 and get sign-off. **Part B** (sections 5–6) runs it.

## 1. Setup
Clones (or updates) the repo and keeps all training runs on Google Drive, so a disconnect never loses a finished run.

In [ ]:
!nvidia-smi --query-gpu=name --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/tooth_seg'
os.makedirs(f'{DRIVE}/runs', exist_ok=True)
os.environ['RUNS_DIR'] = f'{DRIVE}/runs'

GITHUB_USER = 'varshacvenkat-web'
if not os.path.exists('/content/repo'):
    !git clone https://github.com/{GITHUB_USER}/tooth-seg-baseline.git /content/repo
else:
    !cd /content/repo && git pull
%cd /content/repo
!pip install -q -r requirements.txt

## 2. Kaggle token
Add it in the 🔑 Secrets panel as `KAGGLE_API_TOKEN` (notebook access on). Never paste it into a cell.

In [ ]:
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

### Optional: reuse your Week 2 baseline runs
If your Week 2 runs finished all 50 epochs, they were trained with the same settings, split and seeds, so they are copied here and skipped instead of retrained (saves ~55 min). Check the path matches your Drive.

In [ ]:
WEEK2 = '/content/drive/MyDrive/tooth_seg_week2'
for s in [0, 1, 2]:
    src, dst = f'{WEEK2}/baseline_s{s}', f'{DRIVE}/runs/baseline_s{s}'
    if os.path.exists(f'{src}/weights/best.pt') and not os.path.exists(dst):
        !cp -r "{src}" "{dst}"
        print('reusing', src)
!ls "$RUNS_DIR"

## 3. Part A: baseline, sweep, analysis, demo, tests
One command. Without reused runs, about 2 hours on a T4 (3 × 18 min training + ~45 min evaluation and analysis); with them, about 45 minutes. Rerunning skips anything already trained.

In [ ]:
!bash reproduce.sh data baseline analyze demo tests 2>&1 | grep -v "it/s\|s/it\|━"
!mkdir -p "$DRIVE/results" && cp -r results/* "$DRIVE/results/" && cp configs/postprocess.yaml splits.json "$DRIVE/results/"

## 4. Look at the results
These are what you hand in for Part A. Send me the tables and figures and we'll read them together.

In [ ]:
from IPython.display import Markdown, Image, display
for f in ['results/baseline_test.md', 'results/sweep/sweep_val.md', 'results/analysis/ap_by_type.md',
          'results/analysis/per_image_and_pr.md', 'results/failures/failures.md',
          'results/robustness/robustness.md', 'results/integrity.md', 'results/demo/demo.md']:
    display(Markdown(f'### {f}\n\n' + open(f).read()))
for f in ['results/analysis/ap_vs_iou_by_type.png', 'results/analysis/per_image_iou_hist.png',
          'results/failures/gallery_worst20.jpg', 'results/robustness/degradation_examples.jpg']:
    display(Image(f, width=1000))
!tail -2 results/pytest.txt

## ✋ Stop: pre-register Experiment 1
1. Read `results/failures/failures.md`: which failure type is most common, and what is its estimated mAP impact?
2. Pick ONE change aimed at it (`experiments/PREREGISTRATION.md` has a table).
3. Fill in the cell below, share the text with the team, and get sign-off **before** training.
4. Then set `signed_off: True` and run it.

In [ ]:
%%writefile experiments/exp1.yaml
name: exp1
hypothesis: "FILL IN"
change: {imgsz: 1024}      # exactly ONE setting
expected: "FILL IN: metric and failure count you expect to move"
reject_if: "FILL IN: result that rejects the hypothesis"
seeds: [0, 1, 2]
signed_off: false

## 5. Part B: run Experiment 1
Same split, seeds and evaluation; only the one setting changes. An `imgsz: 1024` run takes about 45 min per seed on a T4.

In [ ]:
!bash reproduce.sh exp1 2>&1 | grep -v "it/s\|s/it\|━"
!cp -r results/* "$DRIVE/results/"
display(Markdown(open('results/comparison.md').read()))

## 6. Put it on GitHub
Download these and upload them to your repo (Add file → Upload files), keeping the folder names:
- `splits.json`, `configs/postprocess.yaml`, `experiments/exp1.yaml` (the signed-off version)
- the `results/` folder (skip `results/demo/edge_case_outputs` if it's large)

Then paste the comparison table into the README's Results section.

In [ ]:
!zip -qr /content/week4_results.zip results splits.json configs experiments
from google.colab import files
files.download('/content/week4_results.zip')